In [1]:
import dspy
llama32 = dspy.LM('ollama_chat/llama3.2', api_base='http://localhost:11434', api_key='')
gpt_oss = dspy.LM('ollama_chat/gpt-oss:latest', api_base='http://localhost:11434', api_key='')

# Tools

## Approach 1: Using dspy.ReAct (Fully Managed)

### Basic Example

In [2]:
import dspy

# Define your tools as functions
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # In a real implementation, this would call a weather API
    return f"The weather in {city} is sunny and 75°F"

def search_web(query: str) -> str:
    """Search the web for information."""
    # In a real implementation, this would call a search API
    return f"Search results for '{query}': [relevant information...]"

with dspy.context(lm=llama32):
    # Create a ReAct agent
    react_agent = dspy.ReAct(
        signature="question -> answer",  # Input/output specification
        tools=[get_weather, search_web], # List of available tools
        max_iters=5                      # Maximum number of tool call iterations
    )

    # Use the agent
    result = react_agent(question="What's the weather like in Tokyo?")
    print(result.answer)
    print("Tool calls made:", result.trajectory)

The current weather in Tokyo is sunny and 75°F.
Tool calls made: {'thought_0': "The user is asking about the current weather in Tokyo, so let's start by checking the weather.", 'tool_name_0': 'get_weather', 'tool_args_0': {'city': 'Tokyo'}, 'observation_0': 'The weather in Tokyo is sunny and 75°F', 'thought_1': 'The weather in Tokyo is sunny and 75°F.', 'tool_name_1': 'search_web', 'tool_args_1': {'query': 'Tokyo weather today'}, 'observation_1': "Search results for 'Tokyo weather today': [relevant information...]", 'thought_2': 'The weather in Tokyo is sunny and 75°F.', 'tool_name_2': 'search_web', 'tool_args_2': {'query': 'Tokyo weather today'}, 'observation_2': "Search results for 'Tokyo weather today': [relevant information...]", 'thought_3': 'The weather in Tokyo is sunny and 75°F.', 'tool_name_3': 'search_web', 'tool_args_3': {'query': 'Tokyo weather today'}, 'observation_3': "Search results for 'Tokyo weather today': [relevant information...]", 'thought_4': 'The weather in Tokyo

## Approach 2: Manual Tool Handling

### Basic Setup

In [3]:
import dspy

class ToolSignature(dspy.Signature):
    """Signature for manual tool handling."""
    question: str = dspy.InputField()
    tools: list[dspy.Tool] = dspy.InputField()
    outputs: dspy.ToolCalls = dspy.OutputField()

def weather(city: str) -> str:
    """Get weather information for a city."""
    return f"The weather in {city} is sunny"

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        result = eval(expression)  # Note: Use safely in production
        return f"The result is {result}"
    except:
        return "Invalid expression"

# Create tool instances
tools = {
    "weather": dspy.Tool(weather),
    "calculator": dspy.Tool(calculator)
}

with dspy.context(lm=llama32):
    # Create predictor
    predictor = dspy.Predict(ToolSignature)

    # Make a prediction
    response = predictor(
        question="What's the weather in New York?",
        tools=list(tools.values())
    )

    # Execute the tool calls
    for call in response.outputs.tool_calls:
        # Execute the tool call
        result = call.execute()
        # For versions earlier than 3.0.4b2, use: result = tools[call.name](**call.args)
        print(f"Tool: {call.name}")
        print(f"Args: {call.args}")
        print(f"Result: {result}")

Tool: weather
Args: {'city': 'New York'}
Result: The weather in New York is sunny


### Understanding dspy.Tool

In [4]:
def my_function(param1: str, param2: int = 5) -> str:
    """A sample function with parameters."""
    return f"Processed {param1} with value {param2}"

# Create a tool
tool = dspy.Tool(my_function)

# Tool properties
print(tool.name)        # "my_function"
print(tool.desc)        # The function's docstring
print(tool.args)        # Parameter schema
print(str(tool))        # Full tool description

my_function
A sample function with parameters.
{'param1': {'type': 'string'}, 'param2': {'type': 'integer', 'default': 5}}
my_function, whose description is <desc>A sample function with parameters.</desc>. It takes arguments {'param1': {'type': 'string'}, 'param2': {'type': 'integer', 'default': 5}}.


### Understading dspy.ToolCalls

In [5]:
# After getting a response with tool calls
for call in response.outputs.tool_calls:
    print(f"Tool name: {call.name}")
    print(f"Arguments: {call.args}")

    # Execute individual tool calls with different options:

    # Option 1: Automatic discovery (finds functions in locals/globals)
    result = call.execute()  # Automatically finds functions by name

    # Option 2: Pass tools as a dict (most explicit)
    result = call.execute(functions={"weather": weather, "calculator": calculator})

    # Option 3: Pass Tool objects as a list
    result = call.execute(functions=[dspy.Tool(weather), dspy.Tool(calculator)])

    # Option 4: For versions earlier than 3.0.4b2 (manual tool lookup)
    # tools_dict = {"weather": weather, "calculator": calculator}
    # result = tools_dict[call.name](**call.args)

    print(f"Result: {result}")

Tool name: weather
Arguments: {'city': 'New York'}
Result: The weather in New York is sunny


## Using Native Tool Calling

### Configuration

In [6]:
import dspy

# ChatAdapter with native function calling enabled
chat_adapter_native = dspy.ChatAdapter(use_native_function_calling=True)

# JSONAdapter with native function calling disabled
json_adapter_manual = dspy.JSONAdapter(use_native_function_calling=False)

# Configure DSPy to use the adapter
with dspy.context(lm=llama32, adapter=chat_adapter_native):
    pass

## Async Tools

### Using acall for Async Tools

In [ ]:
import asyncio
import dspy

async def async_weather(city: str) -> str:
    """Get weather information asynchronously."""
    await asyncio.sleep(0.1)  # Simulate async API call
    return f"The weather in {city} is sunny"

with dspy.context(lm=llama32):
    tool = dspy.Tool(async_weather)

    # Use acall for async tools
    result = await tool.acall(city="New York")
    print(result)

The weather in New York is sunny
